# Binary Image Classifier — Improved Transfer-Learning Pipeline

This notebook trains and compares **five ImageNet backbones** (MobileNetV2,
EfficientNetB0, ResNet50, DenseNet121, InceptionV3) as a binary image
classifier, then selects and saves the best one **plus an ensemble**.

It is a hardened rewrite of an earlier baseline. Every change below is either a
**bug fix** or a **research-backed accuracy improvement** — nothing about the
project scope, the dataset, or the folder layout has changed, so it is a
drop-in replacement.

## What changed vs. the baseline, and why

| # | Change | Type | Why it helps |
|---|--------|------|--------------|
| 1 | **Classification head restored to a narrowing funnel** (1280→512→256→1) with position-based layer names | Bug fix | The baseline accidentally swapped two Dense widths (256→512) while leaving stale names; that regressed 4 of 5 backbones (EfficientNetB0 lost >20 F1 points). |
| 2 | **Decision threshold is tuned on validation, then frozen and applied to test** | Accuracy | Every backbone had AUC (0.68–0.84) far above its F1 at a hard-coded 0.5 cut. That gap is pure threshold miscalibration — recovered here for free. |
| 3 | **Two-stage transfer learning: train head frozen, then fine-tune the top layers at a 10× lower LR** (BatchNorm kept frozen) | Accuracy | The baseline *never* unfroze the backbone in any run. Adapting the top layers to this domain is the single biggest lever. |
| 4 | **Final-layer bias initialised to the class log-odds** | Bug fix / stability | Prevents the "always predict one class" collapse the baseline hit with MobileNetV2. |
| 5 | **Single, decoupled weight-decay path (AdamW only)** — removed the duplicate L2 `kernel_regularizer` | Bug fix | The baseline applied weight decay twice via two mechanisms, making the effective regularisation impossible to reason about. |
| 6 | **`.cache()` added to the `tf.data` pipeline** (correct position: after decode, before shuffle) | Speed | Images are decoded/resized once, not every epoch. |
| 7 | **Optional group-aware split hook** (patient/sample-level) | Correctness | Guards against the classic medical-imaging leak where the same subject lands in both train and test. Off by default; see the split cell. |
| 8 | **Probability ensemble of the top backbones** | Accuracy | Diverse CNNs make partially uncorrelated errors; averaging their probabilities is free variance reduction. |
| 9 | **ROC/PR/confusion/training-curve plots + saved artifacts + executive summary** | Reporting | Everything needed to explain the result to a stakeholder. |

> **One thing to decide before a real clinical claim:** does any single
> patient/sample contribute more than one image to `classified_images/`? If yes,
> set `GROUP_SPLIT = True` and fill in `extract_group_id()` in the split cell.

In [ ]:
# Install dependencies (versions pinned for reproducibility). Safe to re-run.
# If your Jupyter server already has these, you can comment this cell out.
# !pip install "tensorflow[and-cuda]==2.17.*" "scikit-learn==1.5.*" \
#     "matplotlib==3.9.*" "pandas==2.2.*" "pillow==10.4.*"

In [ ]:
import os
import json
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

## 1. Configuration

Every knob lives here. Change values in this one cell; the rest of the notebook
reads from it.

In [ ]:
# ------------------------------------------------------------------
# DATA
# ------------------------------------------------------------------
DATA_DIR = "classified_images"        # folder with exactly two class sub-folders

# ------------------------------------------------------------------
# MODELS TO COMPARE
# ------------------------------------------------------------------
MODEL_NAMES = [
    "MobileNetV2",
    "EfficientNetB0",
    "ResNet50",
    "DenseNet121",
    "InceptionV3",
]

# ------------------------------------------------------------------
# IMAGE / BATCH
# ------------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 64

# ------------------------------------------------------------------
# STAGE 1: train the classification head on a FROZEN backbone
# ------------------------------------------------------------------
EPOCHS = 20
LR = 1e-4

# ------------------------------------------------------------------
# STAGE 2: fine-tune the TOP of the backbone at a much lower LR
# ------------------------------------------------------------------
DO_FINETUNE = True            # <- the single biggest accuracy lever
FINE_TUNE_EPOCHS = 15
FINE_TUNE_LR = 1e-5           # 10x lower than stage-1 LR (avoids catastrophic forgetting)
FINE_TUNE_FRACTION = 0.30     # unfreeze the top 30% of backbone layers

# ------------------------------------------------------------------
# REGULARISATION (AdamW decoupled weight decay — the ONLY decay path)
# ------------------------------------------------------------------
WEIGHT_DECAY = 1e-5
DROPOUT_1 = 0.40
DROPOUT_2 = 0.30
USE_CLASS_WEIGHTS = False     # imbalance here is mild (~47/53); threshold tuning handles it

# ------------------------------------------------------------------
# AUGMENTATION
# ------------------------------------------------------------------
USE_HORIZONTAL_FLIP = False   # OFF by default: a flip can invert clinically meaningful
                              # left/right orientation. Turn ON only if laterality is
                              # irrelevant for your images.
ROTATION = 0.05               # +/- ~18 degrees
ZOOM = 0.10
CONTRAST = 0.10

# ------------------------------------------------------------------
# SPLIT
# ------------------------------------------------------------------
VAL_SIZE = 0.15
TEST_SIZE = 0.15
GROUP_SPLIT = False           # set True + fill extract_group_id() if multiple images
                              # can come from the same patient/sample

# ------------------------------------------------------------------
# THRESHOLD SELECTION
# ------------------------------------------------------------------
THRESHOLD_OBJECTIVE = "f1"    # "f1" or "youden" (choose the operating point on VALIDATION)

# ------------------------------------------------------------------
# ENSEMBLE
# ------------------------------------------------------------------
DO_ENSEMBLE = True
ENSEMBLE_TOP_K = 3            # average the top-K models by validation AUC

# ------------------------------------------------------------------
# RUNTIME
# ------------------------------------------------------------------
USE_MIXED_PRECISION = True    # 2-3x faster matmuls on modern NVIDIA GPUs
FORCE_DETERMINISM = False     # True = bit-reproducible but slower (disables fast cuDNN kernels)
SEED = 42
OUTPUT_DIR = "improved_binary_results"
VERBOSE = 1
PROGRESS_EVERY_N_BATCHES = 10

## 2. Reproducibility, GPU and mixed precision

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if FORCE_DETERMINISM:
    # Makes runs bit-for-bit reproducible on GPU, at some speed cost.
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    try:
        tf.config.experimental.enable_op_determinism()
        print("Op determinism enabled.")
    except Exception as e:
        print("Could not enable op determinism:", e)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

for gpu in gpus:
    # Allocate GPU memory incrementally instead of grabbing it all at once
    # (polite on shared machines).
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Could not set memory growth:", e)

if USE_MIXED_PRECISION and len(gpus) > 0:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled (compute float16, variables float32).")
else:
    print("Mixed precision disabled.")

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print("Artifacts will be saved to:", output_dir.resolve())

## 3. Collect image paths

Reads exactly two class sub-folders. Sorting the class folders is load-bearing:
it guarantees the label mapping (`class_0 -> 0`) is identical on every run and
OS, so a metric can never silently flip because the filesystem returned folders
in a different order.

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def collect_image_paths(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Folder not found: {data_dir.resolve()}\n"
            "Expected a folder with exactly two class sub-folders, e.g.:\n"
            "  classified_images/class_0/*.jpg\n"
            "  classified_images/class_1/*.jpg"
        )

    class_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()])
    if len(class_dirs) != 2:
        raise ValueError(
            f"Expected exactly 2 class folders, found {len(class_dirs)}: "
            f"{[p.name for p in class_dirs]}"
        )

    class_names = [p.name for p in class_dirs]
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    image_paths, labels = [], []
    seen = set()
    for class_dir in class_dirs:
        label = class_to_idx[class_dir.name]
        for path in sorted(class_dir.rglob("*")):
            if path.suffix.lower() in VALID_EXTS:
                resolved = str(path.resolve())
                if resolved in seen:      # guard against duplicate/symlinked files
                    continue
                seen.add(resolved)
                image_paths.append(str(path))
                labels.append(label)

    image_paths = np.array(image_paths)
    labels = np.array(labels)
    if len(image_paths) == 0:
        raise ValueError("No images found with extensions " + ", ".join(sorted(VALID_EXTS)))

    return image_paths, labels, class_names, class_to_idx


image_paths, labels, class_names, class_to_idx = collect_image_paths(DATA_DIR)

print("Classes:", class_names)
print("Class mapping:", class_to_idx)
print("Total images:", len(image_paths))
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {class_names[u]}: {c} images ({100 * c / len(labels):.1f}%)")

## 4. Corrupted-image check

`Image.verify()` only checks file structure; a file can still fail when its
pixels are actually decoded mid-training. So we also force a full pixel decode
with `img.load()`. Runs in parallel because it is I/O-bound.

In [ ]:
def _check_one(path):
    try:
        with Image.open(path) as img:
            img.verify()                 # structural check (invalidates the handle)
        with Image.open(path) as img:
            img.load()                   # force full pixel decode
        return None
    except Exception as e:               # noqa: BLE001 - we want any failure
        return (path, str(e))


def find_bad_images(image_paths, max_workers=8):
    bad = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for result in ex.map(_check_one, image_paths):
            if result is not None:
                bad.append(result)
    return bad


bad_files = find_bad_images(image_paths)
print("Bad images found:", len(bad_files))
for path, error in bad_files[:20]:
    print("  ", path, "->", error)

if len(bad_files) > 0:
    bad_set = {p for p, _ in bad_files}
    keep = np.array([p not in bad_set for p in image_paths])
    image_paths, labels = image_paths[keep], labels[keep]
    print("Images remaining after removing bad files:", len(image_paths))

## 5. Train / validation / test split (70 / 15 / 15, stratified)

Two sequential splits with the proportion arithmetic done correctly, so the
final ratio is exactly 70/15/15.

**Group-aware option:** if `GROUP_SPLIT = True`, images are split by *group id*
(e.g. patient) instead of individually, so no subject appears in more than one
split. Customise `extract_group_id()` for your filename/folder convention.

In [ ]:
def extract_group_id(path):
    """Return a group id (e.g. patient/sample id) for one image path.

    CUSTOMISE THIS for your data if GROUP_SPLIT = True. Examples:
      - id is the immediate parent folder under the class folder:
            return Path(path).parent.name
      - id is the filename prefix before the first underscore:
            return Path(path).stem.split("_")[0]
    Default below treats every image as its own group (== a plain random split).
    """
    return str(path)


def make_stratified_splits(image_paths, labels, val_size, test_size, seed):
    temp_size = val_size + test_size
    if not (0 < temp_size < 1):
        raise ValueError("VAL_SIZE + TEST_SIZE must be between 0 and 1.")

    train_p, temp_p, train_y, temp_y = train_test_split(
        image_paths, labels, test_size=temp_size, random_state=seed, stratify=labels
    )
    rel_test = test_size / temp_size
    val_p, test_p, val_y, test_y = train_test_split(
        temp_p, temp_y, test_size=rel_test, random_state=seed, stratify=temp_y
    )
    return train_p, val_p, test_p, train_y, val_y, test_y


def make_group_splits(image_paths, labels, val_size, test_size, seed):
    groups = np.array([extract_group_id(p) for p in image_paths])
    temp_size = val_size + test_size

    gss1 = GroupShuffleSplit(n_splits=1, test_size=temp_size, random_state=seed)
    train_idx, temp_idx = next(gss1.split(image_paths, labels, groups))

    rel_test = test_size / temp_size
    gss2 = GroupShuffleSplit(n_splits=1, test_size=rel_test, random_state=seed)
    val_rel, test_rel = next(
        gss2.split(image_paths[temp_idx], labels[temp_idx], groups[temp_idx])
    )
    val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]

    # Sanity: no group appears in more than one split.
    g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
    assert not (g_tr & g_va), "Group leak: train/val overlap"
    assert not (g_tr & g_te), "Group leak: train/test overlap"
    assert not (g_va & g_te), "Group leak: val/test overlap"

    return (
        image_paths[train_idx], image_paths[val_idx], image_paths[test_idx],
        labels[train_idx], labels[val_idx], labels[test_idx],
    )


if GROUP_SPLIT:
    print("Using GROUP-AWARE split (no subject shared across splits).")
    train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = \
        make_group_splits(image_paths, labels, VAL_SIZE, TEST_SIZE, SEED)
else:
    print("Using stratified image-level split.")
    train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = \
        make_stratified_splits(image_paths, labels, VAL_SIZE, TEST_SIZE, SEED)

for name, y in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    pos = int(np.sum(y == 1))
    print(f"{name}: {len(y)} images | {class_names[1]}={pos} ({100*pos/len(y):.1f}%)")

# Class balance -> optional class weights and -> final-layer bias init.
n_pos = int(np.sum(train_labels == 1))
n_neg = int(np.sum(train_labels == 0))
initial_bias = float(np.log(n_pos / n_neg))     # log-odds; stabilises the sigmoid at t=0
class_weight = None
if USE_CLASS_WEIGHTS:
    total = n_pos + n_neg
    class_weight = {0: total / (2.0 * n_neg), 1: total / (2.0 * n_pos)}
    print("Class weights:", class_weight)
print(f"Final-layer initial bias (log-odds): {initial_bias:.4f}")

## 6. `tf.data` input pipeline

Order matters:
`shuffle(paths)` → `map(decode+resize)` → `cache()` → `batch` → `prefetch`.

- Shuffling **paths** (not decoded images) keeps the shuffle buffer tiny.
- `cache()` after the map means each image is decoded/resized **once ever**,
  not re-decoded every epoch.
- Normalisation is **not** done here — each backbone applies its own
  `preprocess_input` inside the model (different backbones expect different
  normalisation).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
    image = tf.cast(image, tf.float32)          # still raw [0,255]; preprocess happens in-model
    label = tf.cast(label, tf.float32)
    return image, label


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()                             # decode/resize once, reuse every epoch
    if training:
        ds = ds.shuffle(
            buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True
        )
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

for images, batch_labels in train_ds.take(1):
    print("Image batch shape:", images.shape, "| Label batch shape:", batch_labels.shape)

## 7. Model builder

```
Input(224,224,3)
  -> augmentation layers (train-time only, automatic at inference)
  -> backbone-specific preprocess_input
  -> frozen pretrained backbone, global-average-pooled
  -> Dense(512) -> BN -> Dropout(0.4)      <- narrowing funnel (fixed)
  -> Dense(256) -> BN -> Dropout(0.3)
  -> Dense(1, sigmoid, float32, bias=log-odds)
```

Key correctness points:
- The backbone is **called with `training=False`**, so its BatchNorm layers use
  their frozen ImageNet statistics — this holds even during fine-tuning, which
  is exactly what we want.
- Layer names are **positional** (`head_dense_1`, `head_dense_2`), so a future
  width change can never desync a name from the architecture.
- Weight decay comes **only** from AdamW (no duplicate L2 in the layers).

In [ ]:
MODEL_CONFIGS = {
    "MobileNetV2": {
        "model_fn": tf.keras.applications.MobileNetV2,
        "preprocess_fn": tf.keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "model_fn": tf.keras.applications.EfficientNetB0,
        "preprocess_fn": tf.keras.applications.efficientnet.preprocess_input,
    },
    "ResNet50": {
        "model_fn": tf.keras.applications.ResNet50,
        "preprocess_fn": tf.keras.applications.resnet50.preprocess_input,
    },
    "DenseNet121": {
        "model_fn": tf.keras.applications.DenseNet121,
        "preprocess_fn": tf.keras.applications.densenet.preprocess_input,
    },
    "InceptionV3": {
        "model_fn": tf.keras.applications.InceptionV3,
        "preprocess_fn": tf.keras.applications.inception_v3.preprocess_input,
    },
}


def build_model(model_name, initial_bias=0.0):
    config = MODEL_CONFIGS[model_name]
    model_fn = config["model_fn"]
    preprocess_fn = config["preprocess_fn"]

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    # --- augmentation (identity at inference automatically) ---
    x = inputs
    if USE_HORIZONTAL_FLIP:
        x = layers.RandomFlip("horizontal", name="aug_flip")(x)
    x = layers.RandomRotation(ROTATION, name="aug_rotation")(x)
    x = layers.RandomZoom(ZOOM, name="aug_zoom")(x)
    x = layers.RandomContrast(CONTRAST, name="aug_contrast")(x)

    # --- backbone-specific preprocessing ---
    x = layers.Lambda(preprocess_fn, name=f"{model_name}_preprocess")(x)

    # --- pretrained backbone, frozen, global-average-pooled ---
    base_model = model_fn(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg",
    )
    base_model.trainable = False               # stage 1: frozen
    x = base_model(x, training=False)          # BN uses frozen ImageNet stats (always)

    # --- classification head: narrowing funnel ---
    x = layers.Dense(512, activation="relu", name="head_dense_1")(x)
    x = layers.BatchNormalization(name="head_bn_1")(x)
    x = layers.Dropout(DROPOUT_1, name="head_dropout_1")(x)

    x = layers.Dense(256, activation="relu", name="head_dense_2")(x)
    x = layers.BatchNormalization(name="head_bn_2")(x)
    x = layers.Dropout(DROPOUT_2, name="head_dropout_2")(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",                       # keep sigmoid+loss numerically safe under fp16
        bias_initializer=keras.initializers.Constant(initial_bias),
        name="prediction",
    )(x)

    model = keras.Model(inputs, outputs, name=f"{model_name}_binary_classifier")

    # Assert the head shapes match intent (would have caught the baseline's bug).
    assert model.get_layer("head_dense_1").output.shape[-1] == 512
    assert model.get_layer("head_dense_2").output.shape[-1] == 256
    return model, base_model


def unfreeze_top_layers(base_model, fraction):
    """Unfreeze the top `fraction` of backbone layers; keep all BatchNorm frozen."""
    base_model.trainable = True
    n = len(base_model.layers)
    freeze_until = int(n * (1 - fraction))
    trainable_count = 0
    for i, layer in enumerate(base_model.layers):
        if i < freeze_until or isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
            trainable_count += 1
    print(f"  Unfroze {trainable_count}/{n} backbone layers "
          f"(kept the bottom {freeze_until} and all BatchNorm frozen).")

## 8. Training-progress callback (concise, flushes for live logs)

In [ ]:
class BatchProgressCallback(keras.callbacks.Callback):
    def __init__(self, total_epochs, every_n_batches=10, tag=""):
        super().__init__()
        self.total_epochs = total_epochs
        self.every_n_batches = every_n_batches
        self.tag = tag

    def on_epoch_begin(self, epoch, logs=None):
        print(f"\n[{self.tag}] Starting epoch {epoch + 1}/{self.total_epochs}", flush=True)

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        if (batch + 1) % self.every_n_batches == 0:
            msg = f"  batch {batch + 1}"
            for k in ("loss", "accuracy", "auc"):
                if k in logs:
                    msg += f" | {k}: {logs[k]:.4f}"
            print(msg, flush=True)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(
            f"[{self.tag}] epoch {epoch + 1} done | "
            f"loss {logs.get('loss', np.nan):.4f} | acc {logs.get('accuracy', np.nan):.4f} | "
            f"auc {logs.get('auc', np.nan):.4f} || "
            f"val_loss {logs.get('val_loss', np.nan):.4f} | "
            f"val_acc {logs.get('val_accuracy', np.nan):.4f} | "
            f"val_auc {logs.get('val_auc', np.nan):.4f}",
            flush=True,
        )

## 9. Evaluation and threshold selection

The threshold is chosen **on the validation set only**, then frozen and applied
unchanged to the test set — choosing it on test would be leakage.

In [ ]:
def predict_probs(model, dataset):
    """Return (y_true, y_prob). Datasets are unshuffled + cached, so order is stable."""
    y_true = np.concatenate([y.numpy() for _, y in dataset]).astype(int)
    y_prob = model.predict(dataset, verbose=0).reshape(-1).astype(np.float64)
    return y_true, y_prob


def tune_threshold(y_true, y_prob, objective="f1"):
    if len(np.unique(y_true)) < 2:
        return 0.5
    if objective == "youden":
        fpr, tpr, thr = roc_curve(y_true, y_prob)
        j = tpr - fpr
        return float(thr[int(np.argmax(j))])
    # default: maximise F1 on the PR curve
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    # thr has length len(prec)-1; align to it
    best = int(np.argmax(f1[:-1])) if len(thr) > 0 else 0
    return float(thr[best]) if len(thr) > 0 else 0.5


def metrics_at(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        out["roc_auc"] = float("nan")
    return out


def make_callbacks(checkpoint_cb, total_epochs, tag, patience):
    return [
        BatchProgressCallback(total_epochs, PROGRESS_EVERY_N_BATCHES, tag=tag),
        checkpoint_cb,
        keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max",
            patience=patience, restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max",
            factor=0.5, patience=2, min_lr=1e-7, verbose=1,
        ),
    ]

## 10. Train one model (stage 1 frozen head → stage 2 fine-tune → threshold → evaluate)

In [ ]:
def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )


def train_one_model(model_name):
    print("\n" + "=" * 80)
    print(f"Training: {model_name}")
    print("=" * 80)
    keras.backend.clear_session()
    # clear_session() resets the GLOBAL dtype policy back to float32, so the
    # mixed_float16 policy set once at the top would silently NOT apply to any
    # model built after the first clear. Re-assert it here, per model.
    if USE_MIXED_PRECISION and len(gpus) > 0:
        keras.mixed_precision.set_global_policy("mixed_float16")

    model, base_model = build_model(model_name, initial_bias=initial_bias)
    compile_model(model, LR)
    model.summary(line_length=100)

    checkpoint_path = output_dir / f"{model_name}_best.weights.h5"
    # A single checkpoint callback, reused across both stages, so its "best"
    # tracks the global best val_auc over the whole run.
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path), monitor="val_auc", mode="max",
        save_best_only=True, save_weights_only=True, verbose=1,
    )

    start = time.time()

    # ---- Stage 1: frozen backbone, train the head ----
    print("\n--- Stage 1: training head on frozen backbone ---")
    hist1 = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        class_weight=class_weight,
        callbacks=make_callbacks(checkpoint_cb, EPOCHS, tag=f"{model_name}-S1", patience=5),
        verbose=VERBOSE,
    )
    history = {k: list(v) for k, v in hist1.history.items()}
    stage_boundary = len(history.get("loss", []))

    # ---- Stage 2: fine-tune the top of the backbone ----
    if DO_FINETUNE:
        print("\n--- Stage 2: fine-tuning top backbone layers at lower LR ---")
        unfreeze_top_layers(base_model, FINE_TUNE_FRACTION)
        compile_model(model, FINE_TUNE_LR)      # recompile so trainable changes take effect
        hist2 = model.fit(
            train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
            class_weight=class_weight,
            callbacks=make_callbacks(checkpoint_cb, FINE_TUNE_EPOCHS,
                                     tag=f"{model_name}-S2", patience=5),
            verbose=VERBOSE,
        )
        for k, v in hist2.history.items():
            history.setdefault(k, [])
            history[k].extend(list(v))

    training_time = time.time() - start

    # Load the globally-best checkpoint before evaluating.
    if checkpoint_path.exists():
        model.load_weights(checkpoint_path)

    # ---- Predictions ----
    y_val, prob_val = predict_probs(model, val_ds)
    y_test, prob_test = predict_probs(model, test_ds)

    # ---- Threshold: tuned on VAL, frozen, applied to TEST ----
    tuned_thr = tune_threshold(y_val, prob_val, objective=THRESHOLD_OBJECTIVE)

    val_05 = metrics_at(y_val, prob_val, 0.5)
    val_tuned = metrics_at(y_val, prob_val, tuned_thr)
    test_05 = metrics_at(y_test, prob_test, 0.5)
    test_tuned = metrics_at(y_test, prob_test, tuned_thr)   # SAME threshold from val

    print(f"\n{model_name} | tuned threshold = {tuned_thr:.3f}")
    print(f"  VAL  : acc {val_tuned['accuracy']:.3f} | f1 {val_tuned['f1']:.3f} "
          f"| auc {val_tuned['roc_auc']:.3f}")
    print(f"  TEST : acc {test_tuned['accuracy']:.3f} | f1 {test_tuned['f1']:.3f} "
          f"| auc {test_tuned['roc_auc']:.3f}  (vs f1@0.5 {test_05['f1']:.3f})")
    print("  Confusion matrix (test, tuned threshold):")
    print(confusion_matrix(y_test, (prob_test >= tuned_thr).astype(int)))
    print(classification_report(
        y_test, (prob_test >= tuned_thr).astype(int),
        target_names=class_names, zero_division=0,
    ))

    # ---- Save artifacts ----
    pd.DataFrame(history).to_csv(output_dir / f"{model_name}_history.csv", index=False)
    with open(output_dir / f"{model_name}_report.txt", "w") as f:
        f.write(f"Model: {model_name}\nTuned threshold: {tuned_thr:.4f}\n\n")
        f.write("TEST classification report (tuned threshold):\n")
        f.write(classification_report(
            y_test, (prob_test >= tuned_thr).astype(int),
            target_names=class_names, zero_division=0))

    result = {
        "model": model_name,
        "fine_tuned": DO_FINETUNE,
        "training_time_seconds": training_time,
        "total_params": int(model.count_params()),
        "tuned_threshold": tuned_thr,
        "val_auc": val_05["roc_auc"],
        "val_f1_at_0.5": val_05["f1"],
        "val_f1_tuned": val_tuned["f1"],
        "val_acc_tuned": val_tuned["accuracy"],
        "test_auc": test_05["roc_auc"],
        "test_f1_at_0.5": test_05["f1"],
        "test_f1_tuned": test_tuned["f1"],
        "test_acc_tuned": test_tuned["accuracy"],
        "test_precision_tuned": test_tuned["precision"],
        "test_recall_tuned": test_tuned["recall"],
        "checkpoint": str(checkpoint_path),
    }
    # Keep probabilities in memory for the ensemble + plots.
    artifacts = {
        "history": history, "stage_boundary": stage_boundary,
        "y_val": y_val, "prob_val": prob_val,
        "y_test": y_test, "prob_test": prob_test,
        "tuned_threshold": tuned_thr,
    }
    return result, artifacts

## 11. Run the comparison across all backbones

In [ ]:
all_results = []
all_artifacts = {}

for model_name in MODEL_NAMES:
    try:
        result, artifacts = train_one_model(model_name)
        all_results.append(result)
        all_artifacts[model_name] = artifacts
    except Exception as e:                       # keep the batch job alive; print the reason
        print("\n" + "!" * 80)
        print(f"Error while training {model_name}: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        print("Skipping this model.")
        print("!" * 80)

if not all_results:
    raise RuntimeError("No model trained successfully — see the errors above.")

# Rank by validation AUC (threshold-independent → stable across runs).
results_df = pd.DataFrame(all_results).sort_values(
    "val_auc", ascending=False).reset_index(drop=True)
results_df.to_csv(output_dir / "model_comparison.csv", index=False)

pd.set_option("display.max_columns", None, "display.width", 200)
print("\n" + "=" * 80)
print("MODEL COMPARISON (ranked by validation AUC)")
print("=" * 80)
try:
    display(results_df)
except NameError:
    print(results_df.to_string(index=False))

best_model_name = results_df.loc[0, "model"]
print("\nBest single model (by val AUC):", best_model_name)

## 12. Plots — comparison, training curves, ROC / PR, confusion matrix

In [ ]:
# --- 12a. Bar comparison: test AUC vs test F1 (tuned) ---
fig, ax = plt.subplots(figsize=(11, 5))
xpos = np.arange(len(results_df))
w = 0.35
ax.bar(xpos - w / 2, results_df["test_auc"], w, label="Test ROC-AUC")
ax.bar(xpos + w / 2, results_df["test_f1_tuned"], w, label="Test F1 (tuned threshold)")
ax.set_xticks(xpos)
ax.set_xticklabels(results_df["model"], rotation=25, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model comparison on the held-out test set")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "comparison_bars.png", dpi=130)
plt.show()

In [ ]:
# --- 12b. F1 @ 0.5 vs F1 tuned: how much the threshold fix bought us ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(xpos - w / 2, results_df["test_f1_at_0.5"], w, label="Test F1 @ fixed 0.5")
ax.bar(xpos + w / 2, results_df["test_f1_tuned"], w, label="Test F1 @ tuned threshold")
ax.set_xticks(xpos)
ax.set_xticklabels(results_df["model"], rotation=25, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("F1")
ax.set_title("Value of threshold calibration (higher = the fix helped)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "threshold_gain.png", dpi=130)
plt.show()

In [ ]:
# --- 12c. ROC and PR curves for every model ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for model_name, art in all_artifacts.items():
    fpr, tpr, _ = roc_curve(art["y_test"], art["prob_test"])
    auc = roc_auc_score(art["y_test"], art["prob_test"])
    ax1.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")
    prec, rec, _ = precision_recall_curve(art["y_test"], art["prob_test"])
    ax2.plot(rec, prec, label=model_name)
ax1.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax1.set_xlabel("False positive rate"); ax1.set_ylabel("True positive rate")
ax1.set_title("ROC curves (test)"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall curves (test)"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "roc_pr_curves.png", dpi=130)
plt.show()

In [ ]:
# --- 12d. Training curves + confusion matrix for the best model ---
best_art = all_artifacts[best_model_name]
hist = best_art["history"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
if "loss" in hist:
    axes[0].plot(hist["loss"], label="train")
if "val_loss" in hist:
    axes[0].plot(hist["val_loss"], label="val")
axes[0].axvline(best_art["stage_boundary"] - 0.5, color="gray", ls=":", label="fine-tune start")
axes[0].set_title(f"{best_model_name} — loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

if "auc" in hist:
    axes[1].plot(hist["auc"], label="train")
if "val_auc" in hist:
    axes[1].plot(hist["val_auc"], label="val")
axes[1].axvline(best_art["stage_boundary"] - 0.5, color="gray", ls=":")
axes[1].set_title(f"{best_model_name} — AUC"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

thr = best_art["tuned_threshold"]
cm = confusion_matrix(best_art["y_test"], (best_art["prob_test"] >= thr).astype(int))
im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_title(f"{best_model_name} — test confusion\n(threshold={thr:.3f})")
axes[2].set_xticks([0, 1]); axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(class_names); axes[2].set_yticklabels(class_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / f"{best_model_name}_curves_confusion.png", dpi=130)
plt.show()

## 13. Ensemble of the top backbones

Averages the predicted probabilities of the top-K models (by validation AUC).
The ensemble threshold is again tuned on validation and applied to test.

In [ ]:
ensemble_result = None
if DO_ENSEMBLE and len(all_results) >= 2:
    top_k = min(ENSEMBLE_TOP_K, len(results_df))
    members = list(results_df["model"].iloc[:top_k])
    print(f"Ensembling top {top_k} by val AUC: {members}")

    prob_val_stack = np.mean([all_artifacts[m]["prob_val"] for m in members], axis=0)
    prob_test_stack = np.mean([all_artifacts[m]["prob_test"] for m in members], axis=0)
    y_val = all_artifacts[members[0]]["y_val"]
    y_test = all_artifacts[members[0]]["y_test"]

    ens_thr = tune_threshold(y_val, prob_val_stack, objective=THRESHOLD_OBJECTIVE)
    ens_val = metrics_at(y_val, prob_val_stack, ens_thr)     # for model selection (blind to test)
    ens_test = metrics_at(y_test, prob_test_stack, ens_thr)

    print(f"\nENSEMBLE ({'+'.join(members)})")
    print(f"  tuned threshold = {ens_thr:.3f}")
    print(f"  TEST: acc {ens_test['accuracy']:.3f} | f1 {ens_test['f1']:.3f} "
          f"| auc {ens_test['roc_auc']:.3f} | precision {ens_test['precision']:.3f} "
          f"| recall {ens_test['recall']:.3f}")
    print(confusion_matrix(y_test, (prob_test_stack >= ens_thr).astype(int)))

    ensemble_result = {
        "members": members, "tuned_threshold": ens_thr,
        "val_auc": ens_val["roc_auc"],                       # used for selection, not reporting
        "test_auc": ens_test["roc_auc"], "test_f1": ens_test["f1"],
        "test_accuracy": ens_test["accuracy"],
        "test_precision": ens_test["precision"], "test_recall": ens_test["recall"],
    }
else:
    print("Ensemble skipped (need >=2 successful models).")

## 14. Save the winner + executive summary

Picks whichever of *best single model* or *ensemble* wins on **validation AUC**
(so the choice is made blind to the test set), then reports that configuration's
honest test metrics, plus a machine-readable summary and the human summary below.

In [ ]:
best_single = results_df.iloc[0].to_dict()

winner = {"type": "single", "name": best_model_name,
          "test_auc": best_single["test_auc"], "test_f1": best_single["test_f1_tuned"],
          "threshold": best_single["tuned_threshold"]}
if ensemble_result and ensemble_result["val_auc"] > best_single["val_auc"]:
    winner = {"type": "ensemble", "name": "+".join(ensemble_result["members"]),
              "test_auc": ensemble_result["test_auc"], "test_f1": ensemble_result["test_f1"],
              "threshold": ensemble_result["tuned_threshold"]}

summary = {
    "winner": winner,
    "best_single_model": {
        "name": best_model_name,
        "test_auc": float(best_single["test_auc"]),
        "test_f1_at_0.5": float(best_single["test_f1_at_0.5"]),
        "test_f1_tuned": float(best_single["test_f1_tuned"]),
        "test_accuracy_tuned": float(best_single["test_acc_tuned"]),
        "tuned_threshold": float(best_single["tuned_threshold"]),
        "checkpoint": best_single["checkpoint"],
    },
    "ensemble": ensemble_result,
    "config": {
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
        "stage1_epochs": EPOCHS, "fine_tune": DO_FINETUNE,
        "fine_tune_epochs": FINE_TUNE_EPOCHS, "fine_tune_lr": FINE_TUNE_LR,
        "group_split": GROUP_SPLIT, "threshold_objective": THRESHOLD_OBJECTIVE,
        "class_names": class_names,
    },
}
with open(output_dir / "SUMMARY.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

print("=" * 80)
print("EXECUTIVE SUMMARY")
print("=" * 80)
print(f"Dataset            : {len(image_paths)} images | classes {class_names}")
print(f"Split              : {len(train_paths)} train / {len(val_paths)} val / "
      f"{len(test_paths)} test  (group-aware: {GROUP_SPLIT})")
print(f"Backbones compared : {', '.join(MODEL_NAMES)}")
print(f"Fine-tuning        : {'ON' if DO_FINETUNE else 'OFF'}")
print("-" * 80)
print(f"Best single model  : {best_model_name}")
print(f"   test ROC-AUC    : {best_single['test_auc']:.3f}")
print(f"   test F1 @ 0.5   : {best_single['test_f1_at_0.5']:.3f}")
print(f"   test F1 tuned   : {best_single['test_f1_tuned']:.3f}  "
      f"(threshold {best_single['tuned_threshold']:.3f})")
print(f"   test accuracy   : {best_single['test_acc_tuned']:.3f}")
if ensemble_result:
    print("-" * 80)
    print(f"Ensemble           : {'+'.join(ensemble_result['members'])}")
    print(f"   test ROC-AUC    : {ensemble_result['test_auc']:.3f}")
    print(f"   test F1 tuned   : {ensemble_result['test_f1']:.3f}")
    print(f"   test accuracy   : {ensemble_result['test_accuracy']:.3f}")
print("-" * 80)
print(f">>> RECOMMENDED     : {winner['type'].upper()} — {winner['name']}")
print(f"    test AUC {winner['test_auc']:.3f} | test F1 {winner['test_f1']:.3f} "
      f"| operating threshold {winner['threshold']:.3f}")
print("-" * 80)
print(f"All artifacts saved in: {output_dir.resolve()}")
print("  - model_comparison.csv      (full metrics table)")
print("  - SUMMARY.json              (machine-readable summary)")
print("  - *_best.weights.h5         (best weights per backbone)")
print("  - *.png                     (comparison / ROC-PR / confusion plots)")
print("=" * 80)

## 15. How to read this, and what to do next

**How to interpret the numbers**
- **ROC-AUC** measures ranking quality independent of any threshold — use it to
  compare models. **F1 (tuned)** is the score at the chosen operating point.
- The gap between *F1 @ 0.5* and *F1 tuned* is exactly what threshold
  calibration recovered.

**Before trusting these numbers for a real decision**
1. **Confirm there is no subject-level leak.** If any patient/sample can appear
   more than once in `classified_images/`, set `GROUP_SPLIT = True` and fill in
   `extract_group_id()`. Expect the honest numbers to be a little lower — that is
   the point.
2. **Choose the operating threshold by clinical cost**, not F1, if false
   negatives and false positives are not equally costly (set
   `THRESHOLD_OBJECTIVE` or pick a point on the PR curve by hand).
3. **Repeat across a few seeds** (change `SEED`) before declaring a winner — a
   2-3 point F1 gap on a ~400-image test set is within run-to-run noise.